# Diffractive Lens Optimization — Focusing a Plane Wave

Notebook version of `scripts/lens_optimization_example.py`.  We optimize a **phase-only**
element so that a uniform plane wave forms a bright focal spot at a target pixel after
propagation.

This is a compact end-to-end differentiable optics example: a trainable phase mask,
fixed circular aperture, physics-based propagation, and a scalar focusing loss optimized
with Adam.

## Plan:
0. Imports
1. Paths and Parameters
2. Helper Functions
3. Setup
4. Loss Function and Optimization
5. Evaluation
6. Plots



## 0  Imports

We use JAX for automatic differentiation, Optax for optimization, and `fouriax.optics`
for the optical layers and propagation.  The `focal_spot_loss` utility provides a compact
objective for maximizing energy concentration near a desired focal location.


In [ ]:
from __future__ import annotations

from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax

from fouriax.optics import (
    AmplitudeMask,
    Field,
    Grid,
    OpticalModule,
    PhaseMask,
    Spectrum,
    plan_propagation,
)
from fouriax.optim import focal_spot_loss, optimize_optical_module

# NOTEBOOK_REPO_ROOT_SETUP
import os
from pathlib import Path as _Path
%matplotlib inline

def _find_repo_root(start: _Path) -> _Path:
    for candidate in (start, *start.parents):
        if (
            (candidate / "src" / "fouriax").exists()
            and (candidate / "README.md").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate repository root from current working directory. "
        "Expected to find src/fouriax and README.md in an ancestor."
    )

REPO_ROOT = _find_repo_root(_Path.cwd())
if _Path.cwd() != REPO_ROOT:
    os.chdir(REPO_ROOT)


##  1 Paths and Parameters

The notebook resolves the repository root dynamically so that summary JSON and figures are
saved to the repo-level `artifacts/` directory independent of the kernel's working
location.


In [ ]:
ARTIFACTS_DIR = Path("artifacts")
PLOT_PATH = ARTIFACTS_DIR / "lens_optimization_overview.png"
SUMMARY_PATH = ARTIFACTS_DIR / "lens_opt_summary.json"

SEED = 0
GRID_N = 64
GRID_DX_UM = 1.0
WAVELENGTH_UM = 0.532
DISTANCE_UM = 1000.0
APERTURE_DIAMETER_UM = 48.0
WINDOW_PX = 2
LR = 0.05
STEPS = 60


##  2  Helper Functions

We define a binary circular aperture mask

$$
A(x,y) = \begin{cases}
1, & x^2 + y^2 \le (D/2)^2, \\
0, & \text{otherwise},
\end{cases}
$$

which limits the active lens area to diameter `D = aperture_diameter_um`.  The optimized
phase is always multiplied by this fixed aperture before propagation.


In [ ]:
def circular_aperture(grid: Grid, diameter_um: float) -> jnp.ndarray:
    x, y = grid.spatial_grid()
    r2 = x * x + y * y
    radius = diameter_um / 2.0
    return (r2 <= radius * radius).astype(jnp.float32)


## 3  Setup

This section defines the simulation grid, wavelength, propagation distance, and target
focal pixel.  The propagation layer is constructed once using `plan_propagation(mode="auto")`.

The optical pipeline used throughout the notebook is

$$
E_{\mathrm{out}} = \mathcal{P}_z\left\{A(x,y)\,e^{i\phi(x,y)} E_{\mathrm{in}}(x,y)\right\},
$$

where $A(x,y)$ is the circular aperture, $\phi(x,y)$ is the trainable phase, and
$\mathcal{P}_z$ denotes free-space propagation over distance `z = distance_um`.


In [ ]:
grid = Grid.from_extent(nx=GRID_N, ny=GRID_N, dx_um=GRID_DX_UM, dy_um=GRID_DX_UM)
spectrum = Spectrum.from_scalar(WAVELENGTH_UM)
field_in = Field.plane_wave(grid=grid, spectrum=spectrum)

aperture = circular_aperture(grid, diameter_um=APERTURE_DIAMETER_UM)
target_xy = (grid.nx // 2, grid.ny // 2)
propagator = plan_propagation(
    mode="auto",
    grid=grid,
    spectrum=spectrum,
    distance_um=DISTANCE_UM,
)

def build_module(raw_phase_map: jnp.ndarray) -> OpticalModule:
    phase_limited = 2.0 * jnp.pi * jax.nn.sigmoid(raw_phase_map)
    return OpticalModule(
        layers=(
            PhaseMask(phase_map_rad=phase_limited[None, :, :]),
            AmplitudeMask(amplitude_map=aperture[None, :, :]),
            propagator,
        )
    )


## 4 Loss Function and Optimization

### Phase parameterization

We optimize an unconstrained tensor $r(x,y)$ and map it to a bounded phase via

$$
\phi(x,y) = 2\pi\,\sigma(r(x,y)),
$$

where $\sigma$ is the sigmoid.  This keeps the phase in $[0, 2\pi]$ while preserving a
smooth optimization landscape.

### Focusing objective

The propagated field intensity is

$$
I(x,y) = |E_{\mathrm{out}}(x,y)|^2.
$$

`focal_spot_loss(...)` evaluates how well this intensity concentrates around the target
pixel `target_xy`, using a local window of width `window_px`.  Minimizing this loss drives
the phase mask toward a focusing solution.

We initialize the raw phase with small random values and prepare Adam optimizer state.


In [ ]:
def loss_fn(raw_phase_map: jnp.ndarray) -> jnp.ndarray:
    module = build_module(raw_phase_map)
    intensity = module.forward(field_in).intensity()
    return focal_spot_loss(
        intensity=intensity,
        target_xy=target_xy,
        window_px=WINDOW_PX,
    )

key = jax.random.PRNGKey(SEED)
phase_map = 0.1 * jax.random.normal(key, (grid.ny, grid.nx))

optimizer = optax.adam(LR)
result = optimize_optical_module(
    init_params=phase_map,
    build_module=build_module,
    loss_fn=loss_fn,
    optimizer=optimizer,
    steps=STEPS,
    log_every=20,
)
final_phase_limited = 2.0 * jnp.pi * jax.nn.sigmoid(result.best_params)
final_intensity = np.asarray(result.best_module.forward(field_in).intensity())[0]
optimized_profile = final_intensity[target_xy[1], :]


## 5  Evaluation

After optimization, we compute the final focal intensity pattern and compare it to a
classical **hyperbolic-phase lens** reference with phase

$$
\phi_{\mathrm{ref}}(x,y) = -k\left(\sqrt{x^2+y^2+f^2}-f\right),
\qquad k = \frac{2\pi}{\lambda}.
$$

This phase profile equalizes optical path length to a focus at distance `f = distance_um`
(without paraxial approximation), making it a useful baseline for the learned solution.

The notebook compares the optimized and reference center-row intensity profiles.


In [ ]:
x_um, y_um = grid.spatial_grid()
wavelength_um = float(spectrum.wavelengths_um[0])
k = 2.0 * jnp.pi / wavelength_um
hyperbolic_phase = -k * (
    jnp.sqrt(x_um * x_um + y_um * y_um + DISTANCE_UM**2) - DISTANCE_UM
)
reference_module = OpticalModule(
    layers=(
        PhaseMask(phase_map_rad=hyperbolic_phase[None, :, :]),
        AmplitudeMask(amplitude_map=aperture[None, :, :]),
        propagator,
    )
)
reference_intensity = np.asarray(reference_module.forward(field_in).intensity())[0]
reference_profile = reference_intensity[target_xy[1], :]


## 6  Plot Results

In [ ]:
optimized_phase = np.asarray(final_phase_limited)
fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0))

axes[0, 0].plot(result.history)
axes[0, 0].set_title("Loss History")
axes[0, 0].set_xlabel("Step")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].grid(alpha=0.3)

phase_im = axes[0, 1].imshow(optimized_phase, cmap="twilight")
axes[0, 1].set_title("Optimized Phase (rad)")
axes[0, 1].set_xticks([])
axes[0, 1].set_yticks([])
plt.colorbar(phase_im, ax=axes[0, 1], fraction=0.046, pad=0.04)

focus_im = axes[1, 0].imshow(final_intensity, cmap="inferno")
axes[1, 0].set_title("Optimized 2D Focal Spot")
axes[1, 0].set_xticks([])
axes[1, 0].set_yticks([])
plt.colorbar(focus_im, ax=axes[1, 0], fraction=0.046, pad=0.04)

axes[1, 1].plot(optimized_profile, label="Optimized")
axes[1, 1].plot(reference_profile, label="Hyperbolic-phase reference", linestyle="--")
axes[1, 1].axvline(target_xy[0], color="black", linestyle=":", linewidth=1.2, label="Target x")
axes[1, 1].set_title("Center Row Profile")
axes[1, 1].set_xlabel("x pixel")
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

fig.tight_layout()
fig.savefig(PLOT_PATH, dpi=150)
plt.show()
print(f"saved: {PLOT_PATH}")
